# Giai đoạn 12: Stacking Ensemble Learning (Combined Dataset MIMIC + PTB-XL)
Trong giai đoạn này, chúng ta xây dựng kiến trúc **Stacking Ensemble Learning với Cơ chế Dự phòng An toàn (Graceful Fallback)** trên tập dữ liệu gộp MIMIC + PTB-XL (4,230 mẫu):
- **Tầng 1 (Base Models)**: Kết hợp 4 họ mô hình đa dạng: `LightGBM` (Boosting), `Extra Trees` (Bagging), `SVM` (Kernel Boundary) và `TabPFN` (Transformer).
- **Cơ chế An toàn**: Nếu TabPFN gặp sự cố kết nối mạng, hệ thống tự động dự phòng về bộ 3 mô hình Offline (LGB + ET + SVM).
- **Tầng 2 (Meta-Model)**: Sử dụng **Logistic Regression** để tổng hợp xác suất dự đoán.

In [1]:
import os
from dotenv import load_dotenv
for env_path in ['../../.env', '.env', '../.env', 'HealthSense-ML/.env']:
    if os.path.exists(env_path):
        load_dotenv(env_path, override=True)
        break
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.ensemble import StackingClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import joblib
import warnings
warnings.filterwarnings('ignore')

tabpfn_model = None
try:
    tabpfn_token = os.environ.get('TABPFN_TOKEN', 'tabpfn_sk_c3QkRiwEKIItD0qhyM_7N3fvYTJw1x2FAQLBMhQ3kY4')
    os.environ['TABPFN_TOKEN'] = tabpfn_token
    import tabpfn_client
    if tabpfn_token:
        tabpfn_client.set_access_token(tabpfn_token)
    tabpfn_client.init()
    from tabpfn_client import TabPFNClassifier
    tabpfn_model = TabPFNClassifier()
    print('✅ Khởi tạo thành công TabPFN Cloud API!')
except Exception as e:
    try:
        from tabpfn import TabPFNClassifier
        tabpfn_model = TabPFNClassifier(device='cpu')
        print('✅ Khởi tạo thành công TabPFN Local!')
    except Exception as e2:
        print(f'⚠️ Không nạp được TabPFN: {e2}')


✅ Khởi tạo thành công TabPFN Cloud API!


### 1. Nạp và Tiền xử lý dữ liệu Combined (Tập 13 đặc trưng tối ưu)

In [2]:
import os
candidates = [
    '../../data/features/combined_mimic_ptbxl_advanced.csv',
    'data/features/combined_mimic_ptbxl_advanced.csv',
    '../data/features/combined_mimic_ptbxl_advanced.csv'
]
data_path = next((p for p in candidates if os.path.exists(p)), candidates[0])

df_16 = pd.read_csv(data_path)
cols_to_drop = ['LF_HF_Ratio', 'LF_norm', 'HF_norm']
df_13 = df_16.drop(columns=cols_to_drop)

y = df_13['status']
X_13 = df_13.drop(columns=['status'])

X_train, X_test, y_train, y_test = train_test_split(X_13, y, test_size=0.2, random_state=42, stratify=y)
print(f"Bộ dữ liệu Combined: {data_path}")
print(f"Kích thước tập Train: {X_train.shape} | Tập Test: {X_test.shape}")


Bộ dữ liệu Combined: ../../data/features/combined_mimic_ptbxl_advanced.csv
Kích thước tập Train: (3384, 13) | Tập Test: (846, 13)


### 2. Định nghĩa Mô hình Stacking Ensemble & Các Base Models

In [3]:
# 1. Định nghĩa 4 Base Models hợp lực (LightGBM, Extra Trees, SVM, TabPFN)
base_models = [
    ('lightgbm', lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=42, verbose=-1)),
    ('extra_trees', ExtraTreesClassifier(n_estimators=300, random_state=42, class_weight='balanced')),
    ('svm', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('classifier', SVC(probability=True, random_state=42))]))
]

if tabpfn_model is not None:
    base_models.append(('tabpfn', tabpfn_model))
    print('🚀 Đã bao gồm TabPFN vào Hội đồng Stacking 4 Mô hình!')

# 2. Khai báo Meta-Model Regression
meta_model = LogisticRegression(C=1.0, random_state=42)

# 3. Khối Stacking Classifier hoàn chỉnh
stacking_pipeline = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=3,
    n_jobs=1
)


🚀 Đã bao gồm TabPFN vào Hội đồng Stacking 4 Mô hình!


### 3. Huấn luyện & Đánh giá So sánh (Benchmark)

In [4]:
eval_models = {
    "LightGBM": Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('classifier', lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=42, verbose=-1))]),
    "Extra Trees": Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('classifier', ExtraTreesClassifier(n_estimators=300, random_state=42, class_weight='balanced'))]),
    "SVM": Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('classifier', SVC(probability=True, random_state=42))]),
    "Stacking Ensemble (LGB+ET+SVM+TabPFN)": stacking_pipeline
}

if tabpfn_model is not None:
    eval_models["TabPFN"] = tabpfn_model

results = []
confusion_matrices = {}

for name, model in eval_models.items():
    print(f"Đang huấn luyện Combined: {name}...")
    try:
        if name == 'TabPFN':
            X_tr = X_train.values if hasattr(X_train, 'values') else X_train
            X_te = X_test.values if hasattr(X_test, 'values') else X_test
            y_tr = y_train.values if hasattr(y_train, 'values') else y_train
            y_te = y_test.values if hasattr(y_test, 'values') else y_test
            model.fit(X_tr, y_tr)
            y_pred = model.predict(X_te)
        else:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
        
        results.append({
            "Model": name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1-Score": f1_score(y_test, y_pred, zero_division=0)
        })
        confusion_matrices[name] = confusion_matrix(y_test, y_pred)
    except Exception as e:
        print(f"❌ {name} bị lỗi: {e}")

df_results = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False).reset_index(drop=True)
from IPython.display import display
display(df_results)
print('\nBẢNG KẾT QUẢ STACKING ENSEMBLE TRÊN COMBINED:\n')
print(df_results.to_string(index=False))


Đang huấn luyện Combined: LightGBM...


Đang huấn luyện Combined: Extra Trees...


Đang huấn luyện Combined: SVM...


Đang huấn luyện Combined: Stacking Ensemble (LGB+ET+SVM+TabPFN)...


00:00 Fitting... -

...

00:01 Predicting... \

00:02 Predicting... |

00:02 Predicting... Done!


,Model,Accuracy,Precision,Recall,F1-Score
0,TabPFN,0.951537,0.944186,0.959811,0.951934
1,Stacking Ensemble (LGB+ET+SVM+TabPFN),0.947991,0.941725,0.955083,0.948357
2,SVM,0.936170,0.909091,0.969267,0.938215
3,Extra Trees,0.930260,0.915525,0.947991,0.931475
4,LightGBM,0.921986,0.921986,0.921986,0.921986



BẢNG KẾT QUẢ STACKING ENSEMBLE TRÊN COMBINED:

                                Model  Accuracy  Precision   Recall  F1-Score
                               TabPFN  0.951537   0.944186 0.959811  0.951934
Stacking Ensemble (LGB+ET+SVM+TabPFN)  0.947991   0.941725 0.955083  0.948357
                                  SVM  0.936170   0.909091 0.969267  0.938215
                          Extra Trees  0.930260   0.915525 0.947991  0.931475
                             LightGBM  0.921986   0.921986 0.921986  0.921986


### 4. Ghi chú Xuất mô hình

In [5]:
export_path = '../../models/combined_stacking_pipeline.pkl'
os.makedirs('../../models', exist_ok=True)
try:
    joblib.dump(stacking_pipeline, export_path)
    print(f'✅ Đã xuất file mô hình Stacking Ensemble tại: {export_path}')
except Exception as e:
    print(f'⚠️ Không thể xuất file model: {e}')


✅ Đã xuất file mô hình Stacking Ensemble tại: ../../models/combined_stacking_pipeline.pkl
